# KIT sparse RV diagnosis

**Prepared, not executed in Colab.** Use the user's Edge Colab CPU runtime. The original fitted request is diagnosed first. Source and all seven stack repositories are pinned below. KIT v1.1: https://zenodo.org/records/10526554, CC BY-NC 4.0, one volunteer. Two RV volumes provide one initial condition and one subsequent scalar; they are not an RV waveform. All data were already seen. No clinical or independent patient validation claim.

The existing LV candidate is not reconstructed from summary parameters. To inspect it later, regenerate its full selected_request.json using the separate chamber notebook and explicitly supply that path.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
PIN = 'e6db5cc60ed95e233daeb01f53b59abbbc742d02'
ROOT = Path('/content/HeartTwin_RV_Diagnosis_' + PIN[:12])
ROOT.mkdir(exist_ok=True)
LOGS = ROOT/'setup-logs'; LOGS.mkdir(exist_ok=True)
ENV = dict(os.environ, MPLBACKEND='Agg', PYTHONUNBUFFERED='1')
def run(command, name, timeout=2400, cwd=None):
    print('Starting', name, flush=True)
    log_path = LOGS/(name+'.log')
    with log_path.open('w') as log:
        result = subprocess.run(list(map(str, command)), cwd=cwd, env=ENV,
            stdout=log, stderr=subprocess.STDOUT, timeout=timeout)
    if result.returncode:
        print(log_path.read_text()[-10000:])
        raise RuntimeError(name+' failed; retain the log')
PINS = {
 'Virelion-HeartTwin': PIN,
 'Virelion-CardiMech': '5dc14d886ccbda5a1398b863f9ae7821bbffa18b',
 'Virelion-CardiFlow': '670d370635ba09b2908f7e6736299719e86fc36b',
 'Virelion-CardiInfer': 'b4f0c3c30da5986c6a98dba6ceccf1e1de95c881',
 'Virelion-CardiEP': 'f9bfcb78cdb857e5995ce244d8645f7370d6b3bb',
 'Virelion-CardiAnatomy': 'e3cc3450c49b84334dd2da696f33a75f4f61557f',
 'Virelion-ElectroTrace': 'a4cf37ad2e1ed6719a23befa2be5e47af4b4d1de'}
for name, pin in PINS.items():
    path = ROOT/name
    if not path.exists():
        run(['git','init',path], 'init-'+name)
        run(['git','-C',path,'fetch','--depth','1',
             'https://github.com/Virelion-Biotech/'+name+'.git',pin], 'fetch-'+name)
        run(['git','-C',path,'checkout','--detach','FETCH_HEAD'], 'checkout-'+name)
    assert subprocess.check_output(['git','-C',str(path),'rev-parse','HEAD'],text=True).strip()==pin
    assert not subprocess.check_output(['git','-C',str(path),'status','--porcelain','--untracked-files=no'],text=True).strip()
venv = ROOT/'venv'
if not venv.exists():
    run([sys.executable,'-m','venv','--without-pip',venv], 'venv')
python = venv/'bin/python'
ENV['PATH'] = str(python.parent)+os.pathsep+ENV['PATH']
run([sys.executable,'-m','pip','--python',python,'install',
 'pip==25.1.1','numpy==2.2.6','scipy==1.16.3','pydantic==2.12.3',
 'PyYAML==6.0.3','pytest==8.4.2','ruff==0.14.1','meshio==5.3.5',
 'nibabel==5.3.2','pydicom==3.0.1','jsonschema==4.25.1',
 'matplotlib==3.10.7','hypothesis>=6.100','flask>=3.0'], 'dependencies')
for name in PINS:
    run([python,'-m','pip','install','-e',ROOT/name], 'install-'+name)
run([python,'-m','pip','check'], 'pip-check')
run([python,'-m','pip','freeze'], 'environment')
(LOGS/'pins.json').write_text(json.dumps(PINS,indent=2))
repo = ROOT/'Virelion-HeartTwin'
baseline = repo/'validation/kit_edgar_2026_10_20'
OUT = ROOT/'rv-results'; OUT.mkdir(exist_ok=True)
print('Pinned RV diagnostic environment ready')


## Software checks before numerical attribution

A failure must be diagnosed and retained before interpreting physiological sensitivity. The full lifecycle checks remain software/synthetic checks.


In [ ]:
run([python,'-m','py_compile',repo/'scripts/diagnose_kit_rv.py'],
    'rv-runner-compile',cwd=repo)
run([python,'-m','ruff','check',repo/'scripts/diagnose_kit_rv.py'],
    'rv-runner-lint',cwd=repo)
run([python,repo/'scripts/validate_patient_lifecycle.py','--output',ROOT/'lifecycle'],
    'patient-lifecycle',cwd=repo)
print((ROOT/'lifecycle/STATUS.json').read_text())


## Original request: solver, transport and sensitivity

The exact sparse RV timestamp is included in numerical output. The independent solver uses a separate incidence matrix, cosine activation and exprel passive law. Registry and command routes are exercised. No measured times are shifted. The original request remains unchanged.

RUN_DEVELOPMENT_CONSTRAINTS=True explicitly permits fitting to the formerly assessed RV point at three fixed RV outflow resistances. Those results are retrospective development constraints, consume the target for fitting, and cannot count as held-out success. Set False for sensitivity only. Neither setting updates the original or selected LV request.


In [ ]:
REQUEST = baseline/'fitted_model.json'
RUN_DEVELOPMENT_CONSTRAINTS = True
extra = ['--constraint-fit'] if RUN_DEVELOPMENT_CONSTRAINTS else []
run([python,repo/'scripts/diagnose_kit_rv.py','--baseline-dir',baseline,
     '--request',REQUEST,'--output',OUT,*extra], 'rv-diagnosis',cwd=repo)
print((OUT/'solver_transport_checks.json').read_text())
print((OUT/'sensitivity.json').read_text())
if (OUT/'development_constraints.json').exists():
    print((OUT/'development_constraints.json').read_text())
print((OUT/'STATUS.json').read_text())


## Optional: improved LV candidate

Disabled by default. Run the separate chamber notebook first, then enable only when the exact frozen candidate request exists. Missing files cause failure; no silent fallback or partial model reconstruction.


In [ ]:
RUN_SELECTED_LV_CANDIDATE = False
if RUN_SELECTED_LV_CANDIDATE:
    SELECTED_REQUEST = Path('/content/HeartTwin_Chamber_Refinement_ea7cafa73289/results_chamber_final/selected_request.json')
    if not SELECTED_REQUEST.is_file():
        raise RuntimeError('Regenerate the complete selected LV request first')
    run([python,repo/'scripts/diagnose_kit_rv.py','--baseline-dir',baseline,
         '--request',SELECTED_REQUEST,'--output',ROOT/'rv-selected-LV',*extra],
        'rv-selected-LV-diagnosis',cwd=repo)
    print((ROOT/'rv-selected-LV/STATUS.json').read_text())


## Preserve all results and failures

This export includes all generated diagnostic results, conditional requests/predictions, source hashes, logs, test XML and environment. It excludes source geometry. A failed earlier cell does not prevent exporting existing logs: run this cell separately if diagnosis fails. Save the ZIP before closing the runtime. A ZIP alone is not proof of a successful run; inspect STATUS.json and every failure.


In [ ]:
def export_results(root, folders, output_name):
    paths = sorted(p for folder in folders for p in (root/folder).rglob('*') if p.is_file())
    if not paths:
        raise RuntimeError('No result files found; do not create an empty evidence bundle')
    manifest = {str(p.relative_to(root)):hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}
    manifest_path = root/(output_name+'_manifest.json')
    manifest_path.write_text(json.dumps(manifest,indent=2))
    archive = root/(output_name+'.zip')
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as zipped:
        for p in paths+[manifest_path]:
            zipped.write(p,str(p.relative_to(root)))
    print(json.dumps({'archive':str(archive),'bytes':archive.stat().st_size,
       'sha256':hashlib.sha256(archive.read_bytes()).hexdigest(),'file_count':len(paths)},indent=2))
    return archive
archive = export_results(ROOT,['rv-results','rv-selected-LV','lifecycle','setup-logs'],
                         'KIT_RV_diagnosis_results')
from google.colab import files
files.download(str(archive))
